In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

DATA_DIR = Path("../data/mock")
OUTPUT_DIR = Path("../outputs")

c2 = pd.read_csv(
    DATA_DIR / "c2_review_evidence.csv"
)

print("C2 shape:", c2.shape)
display(c2.head(10))

C2 shape: (90, 5)


,review_id,product_id,aspect,evidence_quality,representative_evidence
0,R001,P001,battery,0.94,Battery easily lasts through a full day of nor...
1,R002,P001,battery,0.86,Battery life is generally reliable for daily use.
2,R003,P001,battery,0.72,Heavy use reduces battery life faster than exp...
3,R004,P001,camera,0.88,Photos are detailed in good lighting conditions.
4,R005,P001,camera,0.78,Camera quality is acceptable for everyday use.
5,R006,P001,camera,0.66,Low-light photos can lose detail and clarity.
6,R007,P001,performance,0.95,Apps open quickly and multitasking feels smooth.
7,R008,P001,performance,0.88,Performance remains stable during normal workl...
8,R009,P001,performance,0.74,Long gaming sessions may cause occasional slow...
9,R010,P001,display,0.86,The display is bright and easy to read indoors.


In [2]:
coverage = (
    c2.groupby(["product_id", "aspect"])
    .size()
    .reset_index(name="review_count")
)

display(coverage)

print(
    coverage["review_count"].describe()
)

,product_id,aspect,review_count
0,P001,battery,3
1,P001,camera,3
2,P001,display,3
3,P001,durability,3
4,P001,performance,3
5,P001,price,3
6,P002,battery,3
7,P002,camera,3
8,P002,display,3
9,P002,durability,3


count    30.0
mean      3.0
std       0.0
min       3.0
25%       3.0
50%       3.0
75%       3.0
max       3.0
Name: review_count, dtype: float64


In [3]:
def weighted_bootstrap_evidence(
    evidence_data,
    n_bootstrap=500,
    random_state=42
):
    rng = np.random.default_rng(random_state)

    results = []

    grouped = evidence_data.groupby(
        ["product_id", "aspect"]
    )

    for (product_id, aspect), group in grouped:

        quality = (
            group["evidence_quality"]
            .to_numpy(dtype=float)
        )

        weights = np.clip(
            quality,
            0,
            None
        )

        if weights.sum() == 0:
            probabilities = (
                np.ones(len(weights))
                / len(weights)
            )
        else:
            probabilities = (
                weights / weights.sum()
            )

        for bootstrap_id in range(n_bootstrap):

            sampled_quality = rng.choice(
                quality,
                size=len(quality),
                replace=True,
                p=probabilities
            )

            reliability = (
                sampled_quality.mean()
            )

            results.append({
                "bootstrap_id": bootstrap_id,
                "product_id": product_id,
                "aspect": aspect,
                "evidence_reliability":
                    reliability
            })

    return pd.DataFrame(results)

In [4]:
evidence_bootstrap = (
    weighted_bootstrap_evidence(
        c2,
        n_bootstrap=500,
        random_state=42
    )
)

print(
    "Generated bootstrap rows:",
    len(evidence_bootstrap)
)

display(
    evidence_bootstrap.head(12)
)

Generated bootstrap rows: 15000


,bootstrap_id,product_id,aspect,evidence_reliability
0,0,P001,battery,0.766667
1,1,P001,battery,0.840000
2,2,P001,battery,0.793333
3,3,P001,battery,0.840000
4,4,P001,battery,0.813333
5,5,P001,battery,0.913333
6,6,P001,battery,0.766667
7,7,P001,battery,0.793333
8,8,P001,battery,0.840000
9,9,P001,battery,0.913333


In [5]:
group_counts = (
    evidence_bootstrap
    .groupby(["product_id", "aspect"])
    ["bootstrap_id"]
    .nunique()
)

print(group_counts.describe())

assert (
    group_counts == 500
).all()

print(
    "BOOTSTRAP COUNT TEST: PASSED"
)

count     30.0
mean     500.0
std        0.0
min      500.0
25%      500.0
50%      500.0
75%      500.0
max      500.0
Name: bootstrap_id, dtype: float64
BOOTSTRAP COUNT TEST: PASSED


In [6]:
assert (
    evidence_bootstrap[
        "evidence_reliability"
    ]
    .between(0, 1)
    .all()
)

print(
    "EVIDENCE RANGE TEST: PASSED"
)

EVIDENCE RANGE TEST: PASSED


In [7]:
evidence_summary = (
    evidence_bootstrap
    .groupby(["product_id", "aspect"])
    ["evidence_reliability"]
    .agg(
        mean_reliability="mean",
        evidence_uncertainty="std",
        min_reliability="min",
        max_reliability="max"
    )
    .reset_index()
)

display(
    evidence_summary.head(15)
)

,product_id,aspect,mean_reliability,evidence_uncertainty,min_reliability,max_reliability
0,P001,battery,0.848840,0.052105,0.72,0.94
1,P001,camera,0.783347,0.052118,0.66,0.88
2,P001,display,0.791153,0.041989,0.69,0.86
3,P001,durability,0.797053,0.045759,0.67,0.87
4,P001,performance,0.866513,0.049193,0.74,0.95
5,P001,price,0.785007,0.037313,0.69,0.85
6,P002,battery,0.694520,0.050808,0.56,0.78
7,P002,camera,0.832000,0.053582,0.71,0.93
8,P002,display,0.793480,0.038202,0.69,0.86
9,P002,durability,0.787667,0.046951,0.67,0.87


In [8]:
confidence_intervals = (
    evidence_bootstrap
    .groupby(["product_id", "aspect"])
    ["evidence_reliability"]
    .quantile([0.025, 0.975])
    .unstack()
    .reset_index()
)

confidence_intervals.columns = [
    "product_id",
    "aspect",
    "ci_lower",
    "ci_upper"
]

evidence_summary = (
    evidence_summary.merge(
        confidence_intervals,
        on=["product_id", "aspect"],
        how="left"
    )
)

display(
    evidence_summary.head(15)
)

,product_id,aspect,mean_reliability,evidence_uncertainty,min_reliability,max_reliability,ci_lower,ci_upper
0,P001,battery,0.848840,0.052105,0.72,0.94,0.766667,0.94
1,P001,camera,0.783347,0.052118,0.66,0.88,0.679000,0.88
2,P001,display,0.791153,0.041989,0.69,0.86,0.690000,0.86
3,P001,durability,0.797053,0.045759,0.67,0.87,0.716667,0.87
4,P001,performance,0.866513,0.049193,0.74,0.95,0.762167,0.95
5,P001,price,0.785007,0.037313,0.69,0.85,0.726667,0.85
6,P002,battery,0.694520,0.050808,0.56,0.78,0.606667,0.78
7,P002,camera,0.832000,0.053582,0.71,0.93,0.710000,0.93
8,P002,display,0.793480,0.038202,0.69,0.86,0.726667,0.86
9,P002,durability,0.787667,0.046951,0.67,0.87,0.670000,0.87


In [9]:
highest_uncertainty = (
    evidence_summary
    .sort_values(
        "evidence_uncertainty",
        ascending=False
    )
    .head(10)
)

display(highest_uncertainty)

,product_id,aspect,mean_reliability,evidence_uncertainty,min_reliability,max_reliability,ci_lower,ci_upper
7,P002,camera,0.832000,0.053582,0.71,0.93,0.710000,0.93
13,P003,camera,0.781707,0.052516,0.66,0.88,0.660000,0.88
1,P001,camera,0.783347,0.052118,0.66,0.88,0.679000,0.88
0,P001,battery,0.848840,0.052105,0.72,0.94,0.766667,0.94
12,P003,battery,0.809400,0.051851,0.68,0.90,0.680000,0.90
18,P004,battery,0.811640,0.051670,0.68,0.90,0.680000,0.90
19,P004,camera,0.722667,0.051195,0.60,0.82,0.600000,0.82
25,P005,camera,0.781520,0.051179,0.66,0.88,0.700000,0.88
6,P002,battery,0.694520,0.050808,0.56,0.78,0.606667,0.78
16,P003,performance,0.876700,0.050642,0.75,0.96,0.796667,0.96


In [10]:
controlled_evidence = pd.DataFrame({
    "review_id": [
        "S1", "S2", "S3",
        "V1", "V2", "V3"
    ],

    "product_id": [
        "STABLE", "STABLE", "STABLE",
        "VARIABLE", "VARIABLE", "VARIABLE"
    ],

    "aspect": [
        "battery", "battery", "battery",
        "battery", "battery", "battery"
    ],

    "evidence_quality": [
        0.80, 0.80, 0.80,
        0.30, 0.80, 1.00
    ]
})

controlled_bootstrap = (
    weighted_bootstrap_evidence(
        controlled_evidence,
        n_bootstrap=2000,
        random_state=42
    )
)

controlled_summary = (
    controlled_bootstrap
    .groupby("product_id")
    ["evidence_reliability"]
    .std()
)

display(controlled_summary)

product_id
STABLE      0.000000
VARIABLE    0.135714
Name: evidence_reliability, dtype: float64

In [11]:
stable_std = (
    controlled_summary["STABLE"]
)

variable_std = (
    controlled_summary["VARIABLE"]
)

print(
    "Stable evidence variation:",
    stable_std
)

print(
    "Variable evidence variation:",
    variable_std
)

assert (
    variable_std > stable_std
)

print(
    "EVIDENCE VARIABILITY TEST: PASSED"
)

Stable evidence variation: 0.0
Variable evidence variation: 0.13571429248458786
EVIDENCE VARIABILITY TEST: PASSED


In [12]:
boot_a = weighted_bootstrap_evidence(
    c2,
    n_bootstrap=20,
    random_state=42
)

boot_b = weighted_bootstrap_evidence(
    c2,
    n_bootstrap=20,
    random_state=42
)

assert np.allclose(
    boot_a["evidence_reliability"],
    boot_b["evidence_reliability"]
)

print(
    "BOOTSTRAP REPRODUCIBILITY TEST: PASSED"
)

BOOTSTRAP REPRODUCIBILITY TEST: PASSED


In [13]:
OUTPUT_DIR.mkdir(exist_ok=True)

evidence_bootstrap.to_csv(
    OUTPUT_DIR /
    "evidence_bootstrap_samples.csv",
    index=False
)

evidence_summary.to_csv(
    OUTPUT_DIR /
    "evidence_uncertainty_summary.csv",
    index=False
)

print("Evidence uncertainty outputs saved.")

Evidence uncertainty outputs saved.
